<a href="https://colab.research.google.com/github/C3ZIZ/Nafas-AI/blob/main/Nafas_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required libraries
!pip install librosa matplotlib numpy pandas scikit-learn

import os
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
from scipy.signal import butter, lfilter

# 1. Day 2 Cleaner Logic
def butter_bandpass_filter(data, lowcut=50, highcut=2500, fs=22050, order=5):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut / nyq, highcut / nyq], btype='band')
    return lfilter(b, a, data)

# 2. Day 3 Spectrogram Generator
def generate_mel_spectrogram(segment_data, sr, save_path):
    # Calculate the Mel-Spectrogram
    # n_mels determines the height of the image (number of frequency bands)
    S = librosa.feature.melspectrogram(y=segment_data, sr=sr, n_mels=128, fmax=2500)

    # Convert power to decibels (log scale) for better visual contrast
    S_dB = librosa.power_to_db(S, ref=np.max)

    # Plot and save as an image
    plt.figure(figsize=(4, 4)) # Small square images are great for CNNs
    # We turn off axes to keep only the pure data pattern
    librosa.display.specshow(S_dB, sr=sr, fmax=2500)
    plt.axis('off')
    plt.tight_layout(pad=0)
    plt.savefig(save_path, bbox_inches='tight', pad_inches=0)
    plt.close() # Keep RAM clear

# 3. Processing a single patient's file (Example)
# NOTE: Upload '101_1b1_Al_sc_Meditron.wav' and its .txt file to Colab files before running
def process_and_extract_features(audio_path, txt_path, output_dir="spectrograms"):
    os.makedirs(output_dir, exist_ok=True)

    y, sr = librosa.load(audio_path, sr=22050)
    y_clean = butter_bandpass_filter(y, fs=sr)
    annotations = pd.read_csv(txt_path, sep='\t', header=None, names=['start', 'end', 'crackle', 'wheeze'])

    for i, row in annotations.iterrows():
        start_sample, end_sample = int(row['start'] * sr), int(row['end'] * sr)
        segment = y_clean[start_sample:end_sample]

        # Determine label
        label = "unhealthy" if (row['crackle'] or row['wheeze']) else "healthy"

        # Save the image
        img_name = f"{output_dir}/seg_{i}_{label}.png"
        generate_mel_spectrogram(segment, sr, img_name)
        print(f"Saved: {img_name}")

# Run the function
process_and_extract_features('101_1b1_Al_sc_Meditron.wav', '101_1b1_Al_sc_Meditron.txt')

/tmp/ipykernel_18818/501944372.py:32: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(audio_path, sr=22050)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


FileNotFoundError: [Errno 2] No such file or directory: '101_1b1_Al_sc_Meditron.wav'